In [ ]:
import sys

print("Python utilisé par Jupyter :")
print(sys.executable)
print(sys.version)

In [ ]:
import sys
!"{sys.executable}" -m pip install torch

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import matplotlib
import torch

print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost     :", xgb.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("PyTorch     :", torch.__version__)

print("\n✓ Bibliothèques principales disponibles")

In [ ]:
from pathlib import Path

folder = Path.cwd() / "outputs" / "07D_final_alignment"

print("=== FICHIERS 07D ===")
for f in sorted(folder.iterdir()):
    print(f.name)

In [ ]:
print("\n=== FICHIERS NPY ===")
for f in sorted(folder.glob("*.npy")):
    print(f.name)

In [ ]:
# ============================================================
# NOTEBOOK 08B — LSTM + TCN
# FROZEN PROTOCOL / NO TEST-BASED MODEL SELECTION
#
# Inputs from 07D_final_alignment:
#   sequence X/y/meta, scaled + unscaled
# Target: indoor temperature t+15 min
# Primary sequence representation: 5 x 3
#   5 timesteps = t-60, t-45, t-30, t-15, t
#   channels = temperature, RH, CO2
#
# Selection: validation RMSE only.
# TEST is evaluated once after architecture selection.
# Multiple final training seeds quantify stochastic variability.
# ============================================================

from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
except ImportError as e:
    raise ImportError(
        "PyTorch is required. In Jupyter run:\n"
        "import sys\n!\"{sys.executable}\" -m pip install torch\n"
        "Then restart the kernel and rerun 08B."
    ) from e

ROOT = Path.cwd().resolve()
IN = ROOT / "outputs" / "07D_final_alignment"
OUT = ROOT / "outputs" / "08B_lstm_tcn"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED = {"train": 64701, "validation": 13825, "test": 13790}
SEEDS = [42, 123, 2026]
BATCH_SIZE = 512
MAX_EPOCHS = 60
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*76)
print("NOTEBOOK 08B — LSTM + TCN")
print("="*76)
print("Input :", IN)
print("Output:", OUT)
print("Device:", device)
print("PyTorch:", torch.__version__)

def find_one(patterns):
    for pat in patterns:
        xs = list(IN.glob(pat))
        if len(xs) == 1:
            return xs[0]
    raise FileNotFoundError(f"Could not uniquely find any of: {patterns}")

def files_for(split, scaled=True):

    scale_name = "scaled" if scaled else "unscaled"

    x = IN / f"X_sequence_{split}_{scale_name}.npy"
    y = IN / f"y_sequence_{split}.npy"
    meta = IN / f"metadata_sequence_{split}.csv"

    assert x.exists(), f"Missing X: {x}"
    assert y.exists(), f"Missing y: {y}"
    assert meta.exists(), f"Missing metadata: {meta}"

    return x, y, meta

data = {}
for split in ["train","validation","test"]:
    xp, yp, mp = files_for(split, scaled=True)
    X = np.load(xp).astype(np.float32)
    y = np.load(yp).astype(np.float32).reshape(-1)
    meta = pd.read_csv(mp)
    assert len(X)==len(y)==len(meta)==EXPECTED[split], (split, X.shape, y.shape, len(meta))
    assert X.shape[1:] == (5,3), (split, X.shape)
    data[split] = (X,y,meta)
    print(f"{split:10s}: X={X.shape}, y={y.shape}")

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def metrics(y, p):
    e = p-y
    mae = np.mean(np.abs(e))
    rmse = np.sqrt(np.mean(e**2))
    ssr = np.sum(e**2)
    sst = np.sum((y-y.mean())**2)
    r2 = 1 - ssr/sst
    return dict(n=len(y), MAE=mae, RMSE=rmse, R2=r2,
                Bias=np.mean(e), Error_SD=np.std(e, ddof=1))

class LSTMRegressor(nn.Module):
    def __init__(self, hidden=32, layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(3, hidden, num_layers=layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden,1)
    def forward(self,x):
        z,_ = self.lstm(x)
        return self.head(z[:,-1,:]).squeeze(-1)

class CausalConv1d(nn.Module):
    def __init__(self, cin, cout, kernel=2, dilation=1):
        super().__init__()
        self.pad = (kernel-1)*dilation
        self.conv = nn.Conv1d(cin, cout, kernel, dilation=dilation)
    def forward(self,x):
        x = nn.functional.pad(x, (self.pad,0))
        return self.conv(x)

class TCNRegressor(nn.Module):
    def __init__(self, channels=32, kernel=2, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(3,channels,kernel,1), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(channels,channels,kernel,2), nn.ReLU(), nn.Dropout(dropout)
        )
        self.head = nn.Linear(channels,1)
    def forward(self,x):
        z = self.net(x.transpose(1,2))
        return self.head(z[:,:,-1]).squeeze(-1)

def loader(split, shuffle=False):
    X,y,_=data[split]
    return DataLoader(TensorDataset(torch.from_numpy(X),torch.from_numpy(y)),
                      batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

def predict(model, split):
    model.eval(); ps=[]
    with torch.no_grad():
        for xb,_ in loader(split,False):
            ps.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(ps)

def fit(model, seed, save_path):
    seed_all(seed)
    model=model.to(device)
    opt=torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn=nn.MSELoss()
    best=np.inf; bad=0; history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); losses=[]
        for xb,yb in loader("train",True):
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad()
            loss=loss_fn(model(xb),yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        vp=predict(model,"validation")
        vy=data["validation"][1]
        vrmse=float(np.sqrt(np.mean((vp-vy)**2)))
        history.append({"epoch":epoch,"train_MSE":float(np.mean(losses)),
                        "validation_RMSE":vrmse})
        if vrmse < best - 1e-7:
            best=vrmse; bad=0
            torch.save(model.state_dict(),save_path)
        else:
            bad+=1
            if bad>=PATIENCE: break
    model.load_state_dict(torch.load(save_path,map_location=device,weights_only=True))
    return model,pd.DataFrame(history),best

# Compact, predefined validation-only architecture grid.
# Seed 42 is used for architecture selection. Test remains untouched.
candidates = {
 "LSTM":[
   ("LSTM_h16", lambda:LSTMRegressor(16,1,0.0)),
   ("LSTM_h32", lambda:LSTMRegressor(32,1,0.0)),
   ("LSTM_h64", lambda:LSTMRegressor(64,1,0.0)),
 ],
 "TCN":[
   ("TCN_c16", lambda:TCNRegressor(16,2,0.1)),
   ("TCN_c32", lambda:TCNRegressor(32,2,0.1)),
   ("TCN_c64", lambda:TCNRegressor(64,2,0.1)),
 ]
}

selection=[]
selected={}
print("\n[1/3] VALIDATION-ONLY ARCHITECTURE SELECTION")
for family, cfgs in candidates.items():
    family_rows=[]
    for name, ctor in cfgs:
        p=OUT/f"_selection_{name}.pt"
        model,hist,best=fit(ctor(),42,p)
        row={"family":family,"configuration":name,"selection_seed":42,
             "best_validation_RMSE":best,"epochs_run":len(hist)}
        selection.append(row); family_rows.append(row)
        hist.to_csv(OUT/f"history_selection_{name}.csv",index=False)
        print(row)
    bestrow=min(family_rows,key=lambda r:r["best_validation_RMSE"])
    selected[family]=bestrow["configuration"]

pd.DataFrame(selection).to_csv(OUT/"01_validation_architecture_selection.csv",index=False)
with open(OUT/"02_selected_architectures.json","w") as f:
    json.dump(selected,f,indent=2)
print("Selected:",selected)

ctor_map={name:ctor for cfgs in candidates.values() for name,ctor in cfgs}

print("\n[2/3] FINAL MULTI-SEED TRAINING")
all_metrics=[]; all_preds=[]
for family in ["LSTM","TCN"]:
    cfg=selected[family]
    for seed in SEEDS:
        seed_all(seed)
        model=ctor_map[cfg]()
        ckpt=OUT/f"{family}_seed{seed}.pt"
        t0=time.time()
        model,hist,best=fit(model,seed,ckpt)
        hist.to_csv(OUT/f"history_{family}_seed{seed}.csv",index=False)
        for split in ["train","validation","test"]:
            y=data[split][1]; meta=data[split][2].copy()
            p=predict(model,split)
            m=metrics(y,p)
            all_metrics.append({"model":family,"configuration":cfg,"seed":seed,
                                "split":split,**m,
                                "best_validation_RMSE_during_training":best,
                                "training_seconds":time.time()-t0})
            q=meta.copy()
            q["split"]=split; q["model"]=family; q["seed"]=seed
            q["y_true"]=y; q["y_pred"]=p; q["error"]=p-y
            all_preds.append(q)

metrics_df=pd.DataFrame(all_metrics)
pred_df=pd.concat(all_preds,ignore_index=True)
metrics_df.to_csv(OUT/"03_metrics_all_seeds_all_splits.csv",index=False)
pred_df.to_csv(OUT/"04_predictions_all_seeds.csv",index=False)

summary=(metrics_df.groupby(["model","split"])
         .agg(n_seeds=("seed","nunique"),
              MAE_mean=("MAE","mean"),MAE_SD=("MAE","std"),
              RMSE_mean=("RMSE","mean"),RMSE_SD=("RMSE","std"),
              R2_mean=("R2","mean"),R2_SD=("R2","std"),
              Bias_mean=("Bias","mean"),Bias_SD=("Bias","std"))
         .reset_index())
summary.to_csv(OUT/"05_metrics_mean_sd_across_seeds.csv",index=False)

# Per-device TEST metrics per seed
rows=[]
for (model,seed,uuid),d in pred_df[pred_df.split=="test"].groupby(["model","seed","uuid"]):
    m=metrics(d.y_true.to_numpy(),d.y_pred.to_numpy())
    rows.append({"model":model,"seed":seed,"uuid":uuid,**m})
pd.DataFrame(rows).to_csv(OUT/"06_TEST_per_device_per_seed.csv",index=False)

manifest={
 "protocol_status":"FROZEN_07D_INPUTS",
 "task":"indoor temperature t+15 min",
 "representation":"sequence 5x3",
 "history_minutes":60,
 "step_minutes":15,
 "features":["temperature","relative_humidity","co2"],
 "expected_node_level_counts":EXPECTED,
 "architecture_selection":"validation RMSE only, seed 42",
 "final_training_seeds":SEEDS,
 "test_used_for_selection":False,
 "batch_size":BATCH_SIZE,"max_epochs":MAX_EPOCHS,"patience":PATIENCE,
 "optimizer":"Adam","learning_rate":LR,"weight_decay":WEIGHT_DECAY,
 "device":str(device),"python":platform.python_version(),"torch":torch.__version__,
 "selected_architectures":selected
}
with open(OUT/"07_reproducibility_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)

print("\n[3/3] SUMMARY")
print(summary.to_string(index=False))
print("\nVALIDATION")
print("✓ Frozen 07D sequence samples used")
print("✓ Exact counts 64,701 / 13,825 / 13,790")
print("✓ Primary representation = 5×3 sensor history only")
print("✓ Architecture selected on VALIDATION RMSE only")
print("✓ TEST never used for architecture selection")
print("✓ Three final random seeds: 42, 123, 2026")
print("✓ Predictions saved for paired statistical analysis")
print("✓ Per-device TEST metrics saved")
